# Star-domain vs star-domain interactions

The existing exclusion term in `radially_convex` works **circle-by-circle**: the boundary of set $A$ is pushed back from any circle belonging to set $B$. ...

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import SVG
from jax import numpy as jnp

from vizopt.animation import SnapshotCallback, snapshots_to_animated_svg
from vizopt.base import ObjectiveTerm, OptimConfig, OptimizationProblemTemplate
from vizopt.radially_convex import (
    _build_membership,
    _init_centers_and_radii,
    _multi_term_area,
    _multi_term_enclosure,
    _multi_term_min_radius,
    _multi_term_perimeter,
    _multi_term_smoothness,
    _svg_configuration_fixed,
)


## Random Fourier star domains

In [ ]:
def get_random_radii(thetas, seed):
    rng = np.random.default_rng(seed)
    k = 5
    random_fourier_coeffs = rng.random((k, 2))
    

    mult_thetas = thetas.reshape(-1, 1) * np.arange(1, k + 1)
    logradii = (np.cos(mult_thetas) * random_fourier_coeffs[:, 0]).sum(axis=1) + (
        np.sin(mult_thetas) * random_fourier_coeffs[:, 1]
    ).sum(axis=1)

    radii = logradii + 1 - min(0, min(logradii))
    return radii

thetas = np.linspace(0, 2 * np.pi, 100)
for seed in range(3):
    
    radii = get_random_radii(thetas, seed)

    x = radii * np.cos(thetas)
    y = radii * np.sin(thetas)

    _, ax = plt.subplots(figsize=(5, 5))
    ax.plot(x, y)


### Example: two star domains

In [ ]:
radii_a = get_random_radii(thetas, 0)
radii_b = get_random_radii(thetas, 1)

center_a = (0, 0)
center_b = (6, 6.7)

x_a = center_a[0] + radii_a * np.cos(thetas)
y_a = center_a[1] + radii_a * np.sin(thetas)

x_b = center_b[0] + radii_b * np.cos(thetas)
y_b = center_b[1] + radii_b * np.sin(thetas)

_, ax = plt.subplots(figsize=(5, 5))
ax.scatter(center_a[0], center_a[1])
ax.plot(x_a, y_a)

ax.scatter(center_b[0], center_b[1])
ax.plot(x_b, y_b)
plt.axis("equal")

In [ ]:
tan_theta_a_by_theta_b = (y_b - center_a[1]) / (x_b - center_a[0])
theta_a_by_theta_b = np.arctan(tan_theta_a_by_theta_b)
_, ax = plt.subplots(figsize=(4, 2))
ax.plot(180/np.pi * theta_a_by_theta_b)

There is a collision if and only if min(b_from_center_a - radii_a_by_theta_b) < 0

In [ ]:
_, ax = plt.subplots(figsize=(4, 3))
radii_a_by_theta_b = np.interp(theta_a_by_theta_b, thetas, radii_a)
plt.plot(radii_a_by_theta_b)

b_from_center_a = np.linalg.norm(np.array([x_b - center_a[0], y_b - center_a[1]]).T, axis=1)

plt.plot(b_from_center_a)